# Scraping statistics

All numbers and figures on this page are computed when the book is built, from the files the crawl wrote
to `logs/` (`benchmark.json`, `papers.csv`). The definitions follow the lab book:

- **scraped successfully**: every version of the paper was downloaded and processed, including versions
  that only exist as PDF (they keep an empty record, as the lab book asks);
- **success rate**: over the assigned IDs, and over the papers that actually exist on arXiv
  (part of every assigned range is unused or withdrawn IDs);
- **size before / after**: unpacked size of all files of all versions of a paper, then the size of the
  kept `.tex` and `.bib` files; averaged over papers with at least one LaTeX version;
- **references per paper** and **reference success rate**: from each paper's final `references.json`
  (Semantic Scholar), over the papers on arXiv.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

import report_data as data

data.check()
bench, papers = data.benchmark(), data.papers()
on_arxiv = papers[papers["versions"] > 0]
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})


def pct(value):
    return "–" if value is None else f"{value:.2f}%"

## Summary

In [ ]:
rows = [
    ("Assigned IDs", f"{bench['ids_assigned']:,}"),
    ("Papers that exist on arXiv", f"{bench['papers_on_arxiv']:,}"),
    ("Papers scraped successfully", f"{bench['papers_scraped']:,}"),
    ("Success rate (assigned IDs)", pct(bench["success_rate_of_assigned_pct"])),
    ("Success rate (papers on arXiv)", pct(bench["success_rate_of_papers_on_arxiv_pct"])),
    ("Source versions processed", f"{bench['versions_processed']:,} / {bench['versions_total']:,}"),
    ("Papers with LaTeX source", f"{bench['papers_with_latex']:,}"),
    ("Average paper size before figure removal", f"{bench['avg_paper_bytes_before']:,.0f} B "
                                                  f"({bench['avg_paper_bytes_before'] / 1024:,.1f} KB)"),
    ("Average paper size after (.tex/.bib only)", f"{bench['avg_paper_bytes_after']:,.0f} B "
                                                   f"({bench['avg_paper_bytes_after'] / 1024:,.1f} KB)"),
    ("Space saved", pct(bench["space_saved_pct"])),
    ("Average references per paper", f"{bench['avg_references_per_paper']:.2f}"),
    ("Average arXiv references per paper", f"{bench['avg_arxiv_references_per_paper']:.2f}"),
    ("Reference metadata success rate", pct(bench["reference_success_rate_pct"])),
]
display(pd.DataFrame(rows, columns=["Statistic", "Value"]).style.hide(axis="index"))

## Outcome of every assigned ID

In [ ]:
labels = {"latex": "LaTeX for every version", "mixed": "LaTeX for some versions",
          "pdf_only": "PDF only (empty record)", "incomplete": "a version still missing",
          "not_on_arxiv": "ID not on arXiv"}
counts = papers["outcome"].value_counts().reindex(labels).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.barh([labels[k] for k in counts.index], counts.values, color="#3b7a8f")
ax.bar_label(bars, labels=[f"{v:,} ({v / len(papers):.1%})" for v in counts.values], padding=3)
ax.invert_yaxis()
ax.set_xlabel("Assigned IDs")
ax.set_xlim(0, counts.max() * 1.25)
ax.set_title("What each assigned ID turned out to be")
plt.tight_layout()
plt.show()
display(Markdown(
    f"Assigned IDs: **{len(papers):,}**, of which **{counts['not_on_arxiv']:,}** do not exist on arXiv. "
    f"Papers on arXiv: **{len(on_arxiv):,}**; with LaTeX source: **{counts['latex'] + counts['mixed']:,}**; "
    f"PDF-only: **{counts['pdf_only']:,}** (they keep an empty version folder, so the paper is not dropped); "
    f"still missing a version: **{counts['incomplete']:,}** (listed in `state/download/errors.jsonl`)."))

## Versions per paper

In [ ]:
versions = on_arxiv["versions"].astype(int).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.bar([f"v{v}" for v in versions.index], versions.values, color="#3b7a8f")
ax.bar_label(bars, labels=[f"{v:,}" for v in versions.values], padding=2)
ax.set_xlabel("Number of versions on arXiv")
ax.set_ylabel("Papers")
ax.set_title("Versions per paper")
plt.tight_layout()
plt.show()
single = versions.get(1, 0) / versions.sum()
display(Markdown(
    f"**{single:.1%}** of the papers have a single version; the average is "
    f"**{on_arxiv['versions'].mean():.2f}** versions per paper and the maximum **{int(on_arxiv['versions'].max())}**. "
    f"Every version is downloaded separately (`/src/{{id}}v{{N}}`), so the number of source requests is "
    f"**{int(on_arxiv['versions'].sum()):,}**."))

## Paper size before and after figure removal

In [ ]:
sized = on_arxiv[(on_arxiv["versions_ok"] > 0) & (on_arxiv["size_before"] > 0)]
fig, ax = plt.subplots(figsize=(8, 3.6))
import numpy as np
low = max(1, min(sized["size_after"].min(), sized["size_before"].min()))
bins = np.geomspace(low, sized["size_before"].max() * 1.01, 40)   # log-spaced over the data
ax.hist(sized["size_before"], bins=bins, alpha=0.6, label="before (all files)", color="#c96f53")
ax.hist(sized["size_after"], bins=bins, histtype="step", lw=2, label="after (.tex/.bib)", color="#1f4e5c")
ax.set_xscale("log")
ax.set_xlabel("Size per paper, all versions (bytes, log scale)")
ax.set_ylabel("Papers")
ax.legend()
ax.set_title("Size per paper before and after removing figures")
plt.tight_layout()
plt.show()
display(Markdown(
    f"Over **{len(sized):,}** papers with LaTeX, the median size drops from "
    f"**{sized['size_before'].median() / 1024:,.0f} KB** to **{sized['size_after'].median() / 1024:,.0f} KB**; "
    f"on average **{bench['space_saved_pct']:.1f}%** of every paper is figures and other non-source files "
    f"(images, PDFs, style files), which never reach Google Drive."))

## References per paper

In [ ]:
with_refs = on_arxiv.dropna(subset=["n_refs"])
fig, ax = plt.subplots(figsize=(8, 3.6))
top = max(10, int(with_refs["n_refs"].quantile(0.99)))
bins = range(0, top + 5, 5)
ax.hist(with_refs["n_refs"].clip(upper=top), bins=bins, alpha=0.65, label="all references", color="#3b7a8f")
ax.hist(with_refs["n_arxiv_refs"].clip(upper=top), bins=bins, histtype="step", lw=2,
        label="references with an arXiv ID", color="#b5543a")
ax.set_xlabel(f"References per paper (last bin: {top} or more)")
ax.set_ylabel("Papers")
ax.legend()
ax.set_title("References per paper (Semantic Scholar)")
plt.tight_layout()
plt.show()
share = with_refs["n_arxiv_refs"].sum() / with_refs["n_refs"].sum() if with_refs["n_refs"].sum() else 0
display(Markdown(
    f"Semantic Scholar returned a reference list for **{bench['references_ok']:,}** papers "
    f"({pct(bench['reference_success_rate_pct'])} of the papers on arXiv); **{bench['references_not_found']:,}** "
    f"are not indexed by Semantic Scholar (recent papers can take time to be indexed). On average a paper "
    f"cites **{bench['avg_references_per_paper']:.1f}** works, **{share:.1%}** of them with an arXiv ID; only "
    f"those become rows of the `references` table, but every reference is kept in `references.json`."))